# `max_cpu_time` via the Pyomo `SolverFactory("ripopt")` path

Demonstrates that `solver.options["max_cpu_time"]` truncates a real NLP that would otherwise run several seconds. The problem is circle-packing in the unit square (a classic hard NLP: O(N²) non-overlap constraints, deeply nonconvex), so `N=80` reliably needs more than a second on a laptop and the 1-second budget meaningfully truncates the solve.

`max_cpu_time` is accepted as an alias for `max_wall_time` (issue [#36](https://github.com/jkitchin/ripopt/issues/36)). When the budget fires, ripopt now reports `solver.id == 401` and the message string *"Maximum CPU Time Exceeded"*, matching Ipopt's `Maximum_CpuTime_Exceeded` AMPL convention.

Pyomo's `.sol` parser collapses every solve_result_num in 400–499 to `TerminationCondition.maxIterations`, so that field can't tell a time-out apart from an iter-cap on its own — use `solver.id` and `solver.message` for that.

In [1]:
import pyomo.environ as pyo
import pyomo_ripopt  # noqa: F401
import time

N = 80  # number of circles — bump up if your machine is faster

m = pyo.ConcreteModel()
m.I = pyo.RangeSet(1, N)

def init_x(_, i):
    cols = int(N**0.5) + 1
    return ((i - 1) % cols + 0.5) / cols
def init_y(_, i):
    cols = int(N**0.5) + 1
    return ((i - 1) // cols + 0.5) / cols

m.x = pyo.Var(m.I, bounds=(0, 1), initialize=init_x)
m.y = pyo.Var(m.I, bounds=(0, 1), initialize=init_y)
m.r = pyo.Var(bounds=(1e-3, 0.5), initialize=0.02)

def sep(m, i, j):
    if i >= j:
        return pyo.Constraint.Skip
    return (m.x[i] - m.x[j])**2 + (m.y[i] - m.y[j])**2 >= (2 * m.r)**2
m.sep = pyo.Constraint(m.I, m.I, rule=sep)

m.lo_x = pyo.Constraint(m.I, rule=lambda m, i: m.x[i] >= m.r)
m.hi_x = pyo.Constraint(m.I, rule=lambda m, i: m.x[i] <= 1 - m.r)
m.lo_y = pyo.Constraint(m.I, rule=lambda m, i: m.y[i] >= m.r)
m.hi_y = pyo.Constraint(m.I, rule=lambda m, i: m.y[i] <= 1 - m.r)

m.obj = pyo.Objective(expr=-m.r)  # maximize r

solver = pyo.SolverFactory("ripopt")
solver.options["max_cpu_time"] = 1.0

t0 = time.perf_counter()
result = solver.solve(m)
elapsed = time.perf_counter() - t0

print(f"elapsed:               {elapsed:.2f}s")
print(f"termination_condition: {result.solver.termination_condition}")
print(f"solver.id:             {result.solver.id}")
print(f"solver.message:        {result.solver.message!r}")

model.name="unknown";
    - termination condition: maxIterations
    - message from solver: ripopt 0.8.0\x3a Maximum CPU Time Exceeded


elapsed:               1.10s
termination_condition: maxIterations
solver.id:             401
solver.message:        'ripopt 0.8.0\\x3a Maximum CPU Time Exceeded'
